In [15]:
# ==============================================================================
# 1. INSTALAÇÃO DAS DEPENDÊNCIAS NO AMBIENTE COLAB
# ==============================================================================
!apt-get update -qq > /dev/null
!apt-get install -y -qq chromium-chromedriver > /dev/null
!pip install -q langchain langchain-core langgraph langchain-openai openai-whisper gTTS google-colab-selenium torch torchvision

import os
import io
import time
import base64
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image as PILImage
from IPython.display import HTML, Audio, display, Javascript, Image as IPImage
from google.colab import output

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from gtts import gTTS
import whisper
import google_colab_selenium as gs
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Configure sua chave de API OpenAI (Se não estiver no ambiente, você pode inseri-la diretamente abaixo)
if "OPENAI_API_KEY" not in os.environ or os.environ["OPENAI_API_KEY"] == "SUA_API_KEY_OPENAI_AQUI":
    # Substitua pelo valor de sua chave sk-... se preferir colar diretamente aqui
    os.environ["OPENAI_API_KEY"] = "sk-proj-insira-sua-chave-openai-aqui"

# ==============================================================================
# 2. ETL E TREINAMENTO DA REDE NEURAL PARA FLUXO DE VOZ
# ==============================================================================

# Definição das classes de destino de navegação do vestibular da PUC-Campinas
CLASSES_DESTINO = {
    0: "GUIA_UNIVERSITARIO",      # Apresentação do Guia Universitário e cursos
    1: "MANUAL_DO_CANDIDATO",     # Normas, cronograma e provas
    2: "FORMULARIO_INSCRICAO"     # Área direta de inscrição no portal
}

ROTAS_CONFIG = {
    "GUIA_UNIVERSITARIO": {
        "url": "https://vestibular.puc-campinas.edu.br/guia-universitario/",
        "descricao": "Guia Universitário com apresentação interativa dos cursos e áreas de formação da PUC-Campinas."
    },
    "MANUAL_DO_CANDIDATO": {
        "url": "https://vestibular.puc-campinas.edu.br/manual-do-candidato/",
        "descricao": "Manual Oficial com regras do vestibular, editais e critérios de convocação."
    },
    "FORMULARIO_INSCRICAO": {
        "url": "https://vestibular.puc-campinas.edu.br/",
        "descricao": "Página inicial com acesso direto ao portal de inscrição."
    }
}

# Base de dados textual de comandos de voz (simulando entradas brutas de STT)
dados_brutos = [
    # GUIA_UNIVERSITARIO
    ("mostrar guia do vestibular da puc campinas", 0),
    ("abrir o guia universitario de apresentacao dos cursos", 0),
    ("quero ver a apresentacao do guia da puc campinas", 0),
    ("como funciona o guia de apresentacao para vestibulandos", 0),
    ("navegar ate o guia de cursos e trajetoria universitaria", 0),
    ("mostre a tela do guia interativo de apresentacao", 0),
    # MANUAL_DO_CANDIDATO
    ("abrir manual do candidato com as regras da prova", 1),
    ("onde vejo o edital e as normas do vestibular da puc", 1),
    ("mostrar o manual oficial para leitura das provas", 1),
    ("quero conferir o cronograma e o manual de regras", 1),
    # FORMULARIO_INSCRICAO
    ("como faco para me inscrever no vestibular da puc campinas", 2),
    ("abrir pagina de inscricao para preencher meu cadastro", 2),
    ("ir para a tela de inscricao do vestibular", 2),
    ("quero fazer minha inscricao no vestibular agora", 2)
]

# Construção do vocabulário (Extração e Transformação)
palavras_unicas = sorted(list({p for frase, _ in dados_brutos for p in frase.lower().split()}))
vocab_to_idx = {palavra: i for i, palavra in enumerate(palavras_unicas)}

def vetorizar_frase(frase: str) -> torch.Tensor:
    vetor = [0.0] * len(vocab_to_idx)
    for p in frase.lower().split():
        if p in vocab_to_idx:
            vetor[vocab_to_idx[p]] = 1.0
    return torch.tensor([vetor], dtype=torch.float32)

# Definição do Perceptron Multicamadas (MLP)
class ClassificadorVozPUC(nn.Module):
    def __init__(self, vocab_size: int, hidden_dim: int, num_classes: int):
        super(ClassificadorVozPUC, self).__init__()
        self.rede = nn.Sequential(
            nn.Linear(vocab_size, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.rede(x)

modelo_neural = ClassificadorVozPUC(len(vocab_to_idx), 16, len(CLASSES_DESTINO))
criterio = nn.CrossEntropyLoss()
otimizador = optim.Adam(modelo_neural.parameters(), lr=0.02)

# Treinamento da rede neural (Loop de Ajuste de Pesos)
print("[ETL & Treino] Treinando rede neural para classificação de comandos...")
modelo_neural.train()
for epoch in range(120):
    for frase, classe in dados_brutos:
        otimizador.zero_grad()
        x = vetorizar_frase(frase)
        y = torch.tensor([classe], dtype=torch.long)
        saida = modelo_neural(x)
        loss = criterio(saida, y)
        loss.backward()
        otimizador.step()
modelo_neural.eval()
print("[ETL & Treino] Rede neural treinada e pronta para inferência!")

# ==============================================================================
# 3. FERRAMENTAS DO LANGCHAIN: REDE NEURAL E NAVEGADOR AUTOMATIZADO
# ==============================================================================

@tool
def prever_destino_neural(comando_voz: str) -> str:
    """Classifica o comando transcrito por voz usando a rede neural treinada para identificar a página da PUC-Campinas."""
    with torch.no_grad():
        x = vetorizar_frase(comando_voz)
        logits = modelo_neural(x)
        classe_id = torch.argmax(logits, dim=1).item()
    classe_str = CLASSES_DESTINO[classe_id]
    rota = ROTAS_CONFIG[classe_str]
    return f"CATEGORIA: {classe_str} | URL: {rota['url']} | DESCRICAO: {rota['descricao']}"

@tool
def navegar_e_capturar_tela_puc(url_alvo: str) -> str:
    """Acessa a URL da PUC-Campinas em modo headless, aguarda carregamento, tira screenshot e salva em disco."""
    try:
        driver = gs.Chrome()
        driver.set_window_size(1280, 800)
        driver.get(url_alvo)
        time.sleep(3)  # Aguarda carregamento de scripts dinâmicos da página

        caminho_imagem = "screenshot_puc_campinas.png"
        driver.save_screenshot(caminho_imagem)
        driver.quit()
        return f"Sucesso: Navegação concluída. Imagem salva em {caminho_imagem}."
    except Exception as e:
        return f"Erro durante a navegação web: {str(e)}"

# ==============================================================================
# 4. AGENTE REACT COM LANGGRAPH
# ==============================================================================

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
tools = [prever_destino_neural, navegar_e_capturar_tela_puc]

prompt_sistema = (
    "Você é o Assistente Virtual por Voz do Vestibular da PUC-Campinas. "
    "Sua função é orientar o usuário e acionar a automação para exibir guias e páginas no navegador. "
    "Fluxo obrigatório: "
    "1. Chame 'prever_destino_neural' passando o texto falado pelo usuário para obter a URL correta. "
    "2. Chame 'navegar_e_capturar_tela_puc' usando a URL identificada para carregar a página e tirar a captura de tela. "
    "3. Elabore uma resposta concisa e amigável em português, ideal para ser sintetizada em áudio."
)

# Ajustado o argumento de 'state_modifier' para 'prompt' para compatibilidade com versões recentes do LangGraph
app_agente = create_react_agent(
    model=llm,
    tools=tools,
    prompt=prompt_sistema
)

# ==============================================================================
# 5. PIPELINE DE VOZ (MICROFONE VIA JAVASCRIPT + WHISPER + GTTS)
# ==============================================================================

whisper_model = whisper.load_model("base")

JS_RECORD = """.
var recordVoice = () => {
    return new Promise(resolve => {
        navigator.mediaDevices.getUserMedia({ audio: true }).then(stream => {
            const recorder = new MediaRecorder(stream);
            const chunks = [];
            recorder.ondataavailable = e => chunks.push(e.data);
            recorder.onstop = () => {
                const blob = new Blob(chunks);
                const reader = new FileReader();
                reader.readAsDataURL(blob);
                reader.onloadend = () => resolve(reader.result);
            };
            recorder.start();
            setTimeout(() => recorder.stop(), 5000); // grava por 5 segundos
        });
    });
};
"""

def capturar_voz_colab() -> str:
    print("\n[Microfone Ligado] Fale agora seu pedido (ex: 'mostrar guia do vestibular da puc campinas')...")
    display(Javascript(JS_RECORD))
    audio_b64 = output.eval_js("recordVoice()")
    print("[Áudio Recebido] Transcrevendo voz com Whisper...")

    _, data = audio_b64.split(",", 1)
    audio_bytes = base64.b64decode(data)
    with open("input_voz.webm", "wb") as f:
        f.write(audio_bytes)

    # Ajustado fp16=False para evitar alertas de execução na CPU
    resultado = whisper_model.transcribe("input_voz.webm", language="pt", fp16=False)
    texto_transcrito = resultado["text"].strip()
    return texto_transcrito

def falar(texto: str):
    tts = gTTS(text=texto, lang="pt", tld="com.br", slow=False)
    tts.save("resposta.mp3")
    display(Audio("resposta.mp3", autoplay=True))

# ==============================================================================
# 6. EXECUÇÃO INTEGRADA E TESTE NO COLAB
# ==============================================================================

def rodar_teste_completo_por_voz():
    comando = capturar_voz_colab()
    print(f"\n> Você disse: \"{comando}\"")

    if not comando:
        msg_erro = "Não foi possível identificar o áudio. Por favor, tente novamente."
        print(f"> {msg_erro}")
        falar(msg_erro)
        return

    print("\n[LangGraph] Executando raciocínio com ferramentas neurais e automação web...")
    resultado_agente = app_agente.invoke({"messages": [HumanMessage(content=comando)]})
    resposta_texto = resultado_agente["messages"][-1].content

    print(f"\n[Resposta do Agente]:\n{resposta_texto}\n")

    # Exibe a captura de tela da página da PUC-Campinas gerada pelo Selenium
    if os.path.exists("screenshot_puc_campinas.png"):
        print("[Visualização do Site - PUC-Campinas]:")
        display(IPImage("screenshot_puc_campinas.png", width=750))

    # Emite a resposta por voz
    falar(resposta_texto)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
[ETL & Treino] Treinando rede neural para classificação de comandos...
[ETL & Treino] Rede neural treinada e pronta para inferência!


/tmp/ipykernel_736/3273451990.py:174: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  app_agente = create_react_agent(


In [16]:
rodar_teste_completo_por_voz()


[Microfone Ligado] Fale agora seu pedido (ex: 'mostrar guia do vestibular da puc campinas')...


<IPython.core.display.Javascript object>

MessageError: ReferenceError: recordVoice is not defined